# Phase 5 — Seasonal gap & hybrid suitability

**Phase 5 screening:** Winter wind uses **hub-height DJF percentile** (`wind_hub_speed_p50_DJF`) when available from notebook 01.

Captures NY **summer solar / winter wind** complementarity.

| Input | Output |
|-------|--------|
| `cells_passing_gateway.parquet` | `hybrid_suitability.parquet` |
| Optional: `09`, `10` | `seasonal_gap_report.json`, `maps/hybrid_seasonal_suitability_ny.html` |

**All logic runs in this notebook** (no external script imports).

**Requires:** `01`, `06`. Run after `09`/`10` to attach resource & yield columns.

In [1]:
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import polars as pl
import plotly.graph_objects as go

ROOT = Path('../..').resolve()
GATEWAY_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
RESOURCE_PATH = ROOT / 'data' / 'processed' / 'cell_resource_scores.parquet'
YIELD_PATH = ROOT / 'data' / 'processed' / 'cell_p50_yield.parquet'
OUT_PATH = ROOT / 'data' / 'processed' / 'hybrid_suitability.parquet'
REPORT_PATH = ROOT / 'data' / 'processed' / 'seasonal_gap_report.json'
MAP_PATH = ROOT / 'maps' / 'hybrid_seasonal_suitability_ny.html'
NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'

if not GATEWAY_PATH.exists():
    raise FileNotFoundError(f'Run 06 first: {GATEWAY_PATH}')
print('ROOT:', ROOT)

ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York


In [2]:
df = pl.read_parquet(GATEWAY_PATH).filter(pl.col('in_ny_state'))
print('NY gateway cells:', df.height)

# hub-height winter wind from notebook 01 (fallback to 10 m seasonal mean)
winter_wind_col = (
    'wind_hub_speed_p50_DJF'
    if 'wind_hub_speed_p50_DJF' in df.columns
    else 'wind_speed_mean_DJF'
)
print('Winter wind column:', winter_wind_col)

df = df.with_columns([
    pl.col('shortwave_mean_JJA').alias('summer_ghi'),
    pl.col(winter_wind_col).alias('winter_wind'),
    pl.col('shortwave_mean_DJF').alias('winter_ghi'),
    pl.col('wind_speed_mean_JJA').alias('summer_wind'),
]).with_columns([
    ((pl.col('summer_ghi') - pl.col('winter_ghi')) / pl.col('summer_ghi')).alias('solar_seasonal_range'),
    ((pl.col('winter_wind') - pl.col('summer_wind')) / pl.col('winter_wind')).alias('wind_seasonal_range'),
])

n = df.height
df = df.with_columns([
    (pl.col('summer_ghi').rank() / n).alias('summer_ghi_pct'),
    (pl.col('winter_wind').rank() / n).alias('winter_wind_pct'),
]).with_columns([
    (pl.col('summer_ghi_pct') * pl.col('winter_wind_pct')).alias('hybrid_seasonal_score'),
])

q_summer = df['summer_ghi_pct'].quantile(0.75)
q_winter = df['winter_wind_pct'].quantile(0.75)
df = df.with_columns(
    ((pl.col('summer_ghi_pct') >= q_summer) & (pl.col('winter_wind_pct') >= q_winter)).alias('jackpot_zone')
)
df.head(3)

NY gateway cells: 1089
Winter wind column: wind_hub_speed_p50_DJF


latitude,longitude,shortwave_mean,shortwave_mean_24h,shortwave_interannual_std,shortwave_intraseasonal_std_mean,direct_mean,diffuse_mean,shortwave_diurnal_amplitude,clear_sky_index,direct_diffuse_ratio,wind_speed_mean,wind_speed_interannual_std,wind_speed_intraseasonal_std_mean,wind_speed_p90,wind_power_mean,wind_speed_hub_mean,wind_speed_hub_std,wind_hub_power_mean,wind_hub_speed_p50,wind_hub_speed_p75,wind_hub_speed_p90,wind_hub_speed_p95,wind_ramp_p95,weibull_k,weibull_A,temperature_mean,surface_pressure_mean,n_years,year_start,year_end,shortwave_mean_m01,shortwave_mean_m02,shortwave_mean_m03,shortwave_mean_m04,shortwave_mean_m05,shortwave_mean_m06,…,frac_class_43,frac_class_52,frac_class_71,frac_class_81,frac_class_82,frac_class_90,frac_class_95,dist_to_transmission_km,in_ny_state,frac_agriculture,frac_barren,frac_dev_open,frac_forest,frac_wetland,frac_developed,frac_water,frac_ice,developability_raw,developability_D,fail_water,fail_developed,fail_wetland,fail_ice,fail_low_D,fail_no_nlcd,fail_population,passes_gateway,summer_ghi,winter_wind,winter_ghi,summer_wind,solar_seasonal_range,wind_seasonal_range,summer_ghi_pct,winter_wind_pct,hybrid_seasonal_score,jackpot_zone
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32,i32,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
42.39,-79.26,307.224073,163.16216,7.925794,244.916909,102.171323,60.990837,981.625,0.456804,1.666013,3.608008,0.220882,1.756773,6.0143072,85.120093,4.827254,2.350435,203.858998,4.508429,6.281873,8.046707,9.14822,1.621185,2.17523,5.448487,8.7627,961.933439,8,2018,2025,137.859845,214.098508,287.820831,346.399853,382.525785,416.482395,…,0.108913,0.004667,0.001891,0.239216,0.033062,0.03024,0.000714,1.674329,true,0.272279,0.001212,0.056626,0.597857,0.030954,0.084154,0.006986,0.0,0.020139,0.470536,false,false,false,false,false,false,false,true,409.950399,5.350723,158.269954,2.895852,0.613929,0.458792,0.988522,0.902204,0.891848,true
41.94,-73.77,305.184693,163.072233,10.555213,240.311179,104.059784,59.012449,960.5,0.462158,1.766609,3.002355,0.171541,1.437081,4.864987,47.851187,4.016933,1.92271,114.601555,3.882103,5.170032,6.509,7.431065,1.665604,2.206944,4.535318,10.403612,991.197862,8,2018,2025,183.88871,233.707862,312.967387,328.688167,365.663666,382.812601,…,0.069532,0.00205,0.003051,0.120864,0.000059,0.068215,0.006201,2.422054,true,0.120923,0.000164,0.077935,0.695438,0.074416,0.097716,0.006242,0.0,-0.208157,0.29739,false,false,false,false,false,false,false,true,382.13685,4.231579,190.804741,2.585639,0.50069,0.388966,0.523416,0.266299,0.139385,false
43.11,-73.86,298.392867,159.534293,10.697904,236.33392,100.345413,59.18888,960.375,0.458003,1.733233,2.903641,0.132312,1.429785,4.807198,45.042919,3.884862,1.912948,107.875872,3.655904,4.930027,6.431683,7.471011,1.515546,2.146063,4.384753,8.966889,991.822526,8,2018,2025,173.306538,231.86126,307.717433,322.956955,361.002342,375.384115,…,0.227885,0.009258,0.010113,0.058879,0.002448,0.183341,0.008362,1.393111,true,0.061326,0.013462,0.086892,0.520638,0.191703,0.192353,0.001148,0.0,-0.301794,0.226372,false,false,false,false,false,false,false,true,375.871283,3.738151,182.296824,2.57395,0.515002,0.311438,0.177227,0.121212,0.021482,false


In [3]:
if RESOURCE_PATH.exists():
    df = df.join(pl.read_parquet(RESOURCE_PATH), on=['latitude', 'longitude'], how='left')
if YIELD_PATH.exists():
    y = pl.read_parquet(YIELD_PATH).select(['latitude', 'longitude', 'solar_cf_p50', 'wind_cf_p50', 'hybrid_cf_p50'])
    df = df.join(y, on=['latitude', 'longitude'], how='left')

keep = [
    'latitude', 'longitude', 'summer_ghi', 'winter_wind', 'winter_ghi', 'summer_wind',
    'solar_seasonal_range', 'wind_seasonal_range', 'summer_ghi_pct', 'winter_wind_pct',
    'hybrid_seasonal_score', 'jackpot_zone',
]
extra = [c for c in df.columns if c not in keep and ('screening' in c or 'resource_' in c or '_cf_p50' in c)]
df = df.select(keep + extra)
df.write_parquet(OUT_PATH)
print(f'Wrote {OUT_PATH}')

Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/hybrid_suitability.parquet


In [4]:
report = {
    'n_cells': df.height,
    'winter_wind_column': winter_wind_col,
    'mean_summer_ghi_JJA': float(df['summer_ghi'].mean()),
    'mean_winter_ghi_DJF': float(df['winter_ghi'].mean()),
    'mean_solar_seasonal_drop_frac': float(1 - df['winter_ghi'].mean() / df['summer_ghi'].mean()),
    'mean_winter_wind_DJF': float(df['winter_wind'].mean()),
    'mean_summer_wind_JJA': float(df['summer_wind'].mean()),
    'mean_wind_winter_lift_frac': float(df['winter_wind'].mean() / df['summer_wind'].mean() - 1),
    'n_jackpot_zones': int(df['jackpot_zone'].sum()),
    'top_hybrid_seasonal': df.sort('hybrid_seasonal_score', descending=True).head(5).to_dicts(),
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(f"Solar winter drop (avg): {report['mean_solar_seasonal_drop_frac']:.1%}")
print(f"Wind winter lift (avg): {report['mean_wind_winter_lift_frac']:.1%}")
print(f"Jackpot zones: {report['n_jackpot_zones']} / {report['n_cells']}")

Solar winter drop (avg): 56.2%
Wind winter lift (avg): 69.1%
Jackpot zones: 143 / 1089


In [5]:
pdf = df.to_pandas()
ny = gpd.read_file(NY_BOUNDARY_PATH)
outline = ny.boundary.iloc[0]

fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=pdf['latitude'], lon=pdf['longitude'], mode='markers',
    marker=dict(size=8, color=pdf['hybrid_seasonal_score'], colorscale='Viridis', showscale=True, opacity=0.7),
))
jack = pdf[pdf['jackpot_zone']]
fig.add_trace(go.Scattermap(
    lat=jack['latitude'], lon=jack['longitude'], mode='markers',
    marker=dict(size=11, color='crimson'), name='Jackpot zones',
))
fig.add_trace(go.Scattermap(
    lat=list(outline.xy[1]), lon=list(outline.xy[0]),
    mode='lines', line=dict(width=2, color='#333'), hoverinfo='skip', showlegend=False,
))
fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=42.9, lon=-75.8), zoom=5.5),
    title='Hybrid seasonal suitability (summer GHI × winter wind percentiles)',
    height=800,
)
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(MAP_PATH), include_plotlyjs='cdn')
print(f'Map -> {MAP_PATH}')

Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/hybrid_seasonal_suitability_ny.html
